# Lab 09: Assignment 2 Tutorial

In Assignment 1, we followed a wave packet in a harmonic well. Now put it in the right-hand well of a double-well potential.

A classical particle below the barrier cannot cross. What will the quantum probability density do, and how can we tell when it has reached the left well?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 13, "figure.dpi": 100})

We'll use $\alpha=0.4$ here. Use your assigned value in the homework.

## Two wells and a barrier

The potential in Assignment 2 is

$$
V(x)=\alpha x^4-2x^2+\frac{1}{\alpha}.
$$

The parameter $\alpha$ changes both the barrier height and the distance between the wells.

In [ ]:
def V(x, alpha):
    return alpha * x**4 - 2 * x**2 + 1 / alpha

The minima are at $\pm x_{\min}=\pm1/\sqrt{\alpha}$, where $V=0$. What is the barrier height $V(0)$?

## Put the wavefunction on a grid

We will store one complex amplitude at each position. Use $m=\hbar=1$ and 600 intervals on $[-4,4]$.

In [ ]:
ALPHA = 0.4
HBAR = 1
BOXSIZE = 8
ND = 600
DELTAX = BOXSIZE / ND
XMIN = 1 / np.sqrt(ALPHA)
x = np.linspace(-BOXSIZE / 2, BOXSIZE / 2, ND + 1)
print("Positions:", x.size, "Spacing:", DELTAX)

There are 601 positions and 600 intervals. The spacing $\Delta x$ will weight our spatial sums.

## Change the well

In [ ]:
alpha_view = 0.4

fig, ax = plt.subplots(
    figsize=(6.2, 3.6),
    layout="constrained")
ax.plot(x, V(x, alpha_view), color="black")
ax.set(xlabel="x", ylabel="V(x)",
       xlim=(-4, 4), ylim=(0, 6),
       title=f"alpha = {alpha_view}")
plt.show()

Increase `alpha_view` to 0.6 and rerun. Does the barrier get taller or shorter? Do the wells move closer?

## Choose a state in one well

Near either minimum, the potential is approximately quadratic. That gives us a familiar starting point: the harmonic oscillator.

[Lecture 10, slide 4](https://jduarte.physics.ucsd.edu/phys142/lectures/10_DoubleWellTunneling.pdf#page=4) gives

$$
V_+(x)=4(x-x_{\min})^2,\qquad V_-(x)=4(x+x_{\min})^2.
$$

In [ ]:
def V_plus(x, alpha):
    return 4 * (x - 1 / np.sqrt(alpha))**2

def V_minus(x, alpha):
    return 4 * (x + 1 / np.sqrt(alpha))**2

These parabolas describe the neighborhoods of the two minima. They need not agree with the full potential between the wells.

## How local is the approximation?

In [ ]:
alpha_view = 0.4
x_window = (-4, 4)

fig, ax = plt.subplots(
    figsize=(6.2, 3.6),
    layout="constrained")
ax.plot(x, V(x, alpha_view), "k", label="V")
ax.plot(x, V_plus(x, alpha_view), "--",
        label="V+")
ax.plot(x, V_minus(x, alpha_view), "--",
        label="V-")
ax.set(xlabel="x", ylabel="Potential",
       xlim=x_window, ylim=(0, 6))
ax.legend()
plt.show()

Set `x_window = (0.8, 2.3)` to inspect the right well. Where does the approximation start to fail?

## The initial wavefunction

The local oscillator has $\omega=2\sqrt{2}$. Its ground state is a Gaussian. Center it at the right minimum to put most of the probability in that well:

$$
\psi(x)=\left(\frac{\omega}{\pi}\right)^{1/4}
\exp\left[-\frac{\omega}{2}(x-x_c)^2\right].
$$

In [ ]:
OMEGA = np.sqrt(8)

def func_psi_0(x, x_start):
    factor = (OMEGA / np.pi)**0.25
    return factor * np.exp(
        -OMEGA * (x - x_start)**2 / (2 * HBAR))

This function returns an amplitude. The probability density is $|\psi(x)|^2$.

## Place the initial state

In [ ]:
center = XMIN

fig, ax = plt.subplots(
    figsize=(6.2, 3.6),
    layout="constrained")
ax.plot(x, func_psi_0(x, center))
ax.axvline(center, color="gray",
           linestyle=":")
ax.set(xlabel="x", ylabel="Wavefunction",
       xlim=(-4, 4), ylim=(0, 1.1),
       title=f"Center = {center:.2f}")
plt.show()

Set `center = -XMIN` and rerun. Which well now contains most of the probability?

## Why can a localized state move?

The full potential is symmetric. Its energy eigenstates can be chosen even or odd, but our right-well Gaussian is neither.

$$
\psi_{\mathrm S}=\frac{\psi_++\psi_-}{\sqrt{2}},\qquad
\psi_{\mathrm A}=\frac{\psi_+-\psi_-}{\sqrt{2}}.
$$

In [ ]:
psi_plus = func_psi_0(x, XMIN)
psi_minus = func_psi_0(x, -XMIN)
psi_S = (psi_plus + psi_minus) / np.sqrt(2)
psi_A = (psi_plus - psi_minus) / np.sqrt(2)

These approximate the lowest even and odd states. Different energies give different phase rates, allowing a superposition to change its density.

## Compare the two combinations

In [ ]:
fig, ax = plt.subplots(
    figsize=(6.2, 3.6),
    layout="constrained")
ax.plot(x, psi_S, label="Symmetric")
ax.plot(x, psi_A, "--",
        label="Antisymmetric")
ax.axhline(0, color="gray", linewidth=0.7)
ax.set(xlabel="x", ylabel="Wavefunction",
       xlim=(-4, 4), ylim=(-0.9, 0.9))
ax.legend()
plt.show()

Which combination changes sign under $x\to-x$? Which has a node at the barrier?

## Advance the amplitudes

The initial state tells us where the probability starts. The propagator advances its complex amplitudes through a short time step. Using the midpoint potential,

$$
\tilde{\mathcal K}(x_b,\Delta t;x_a,0)=
\exp\left[i\left(\frac{(x_b-x_a)^2}{2\Delta t}
-V\left(\frac{x_a+x_b}{2}\right)\Delta t\right)\right].
$$

In [ ]:
def func_K(x_a, x_b, dt, alpha):
    kinetic = 0.5 * (x_b - x_a)**2 / dt
    potential = V((x_a + x_b) / 2, alpha) * dt
    return np.exp(1j * (kinetic - potential))

The first term in the exponent comes from kinetic energy; the second samples the potential halfway between the endpoints.

## One starting point, many destinations

In [ ]:
dt_view = np.pi / 128
start = XMIN
x_b = start + np.linspace(-0.6, 0.6, 401)
amplitude = func_K(start, x_b, dt_view, ALPHA)
fig, ax = plt.subplots(figsize=(6.2, 3.6),
                       layout="constrained")
ax.plot(x_b, amplitude.real,
        label="Real")
ax.plot(x_b, amplitude.imag, "--",
        label="Imag")
ax.set(xlabel="Destination x",
       ylabel="Kernel",
       ylim=(-1.1, 1.1))
ax.legend()
plt.show()

Halve `dt_view` and rerun. Do the oscillations become faster or slower? This is a complex amplitude, not a probability.

## From the propagator to the next state

Add the complex amplitudes from every starting position. On the grid, the integral becomes a weighted sum:

$$
\begin{aligned}
\psi(x_b,t+\Delta t)&\approx\int \mathcal K(x_b,x_a;\Delta t)\psi(x_a,t)\,dx_a,\\
\psi_i(t+\Delta t)&\approx\Delta x\sum_j K_{ij}\psi_j(t).
\end{aligned}
$$

Why do we add the amplitudes before taking the absolute square?

## Build the matrix

Each column starts at one position; each row arrives at one position.
Use the assignment's $\Delta t=\pi/128$ for the evolution.

In [ ]:
DELTAT = np.pi / 128
K_dt = np.zeros((ND + 1, ND + 1), dtype=np.complex64)

for i in range(ND + 1):
    for j in range(ND + 1):
        K_dt[i, j] = func_K(x[j], x[i], DELTAT, ALPHA)

print("Kernel shape:", K_dt.shape)

For a fixed starting position $j$, which slice gives all of its destination amplitudes?

## Take one step

The grid sum is a matrix–vector product. Normalize after each step because our kernel leaves out the scalar normalization factor.

In [ ]:
def step(state, kernel, dx):
    updated = dx * (kernel @ state)
    norm = dx * np.sum(updated * updated.conjugate())
    return updated / np.sqrt(norm)

psi_0 = func_psi_0(x, XMIN)
print("Initial norm:",
      DELTAX * np.sum(np.abs(psi_0)**2))

Normalization keeps the total probability at one. It does not by itself establish that the evolution is accurate.

## What changes after a few steps?

In [ ]:
steps_view = 20
state_view = psi_0.copy()
for _ in range(steps_view):
    state_view = step(
        state_view, K_dt, DELTAX)

fig, ax = plt.subplots(figsize=(6.2, 3.6),
                       layout="constrained")
ax.plot(x, np.abs(psi_0)**2, "--",
        label="Initial")
ax.plot(x, np.abs(state_view)**2,
        label="Evolved")
ax.set(xlabel="x",
       ylabel="Probability density",
       xlim=(-4, 4), ylim=(0, 1.3))
ax.legend()
plt.show()

Try `steps_view = 100`. Does the packet translate without changing shape?

## Follow the motion

Run to $T=20\pi$ and keep the states so we can inspect them afterward.
Store the initial state as step 0.

In [ ]:
T0 = 20 * np.pi
NT = int(T0 / DELTAT)
t = np.arange(NT + 1) * DELTAT

psi = [psi_0]
for _ in range(NT):
    psi.append(step(psi[-1], K_dt, DELTAX))

prob = np.abs(np.asarray(psi))**2
print("Stored states:", len(psi))

Why are there $N_T+1$ stored states for $N_T$ time steps?

## Measure the transfer

The mean position is the probability-weighted average of position:

$$
\langle x\rangle(t)\approx\Delta x\sum_i x_i|\psi(x_i,t)|^2.
$$

It starts near the right minimum. Its first major minimum marks the first trip toward the left well.

Would $\langle x\rangle=0$ tell us that the particle is at the barrier?

## Follow the mean position

In [ ]:
x_bar = DELTAX * (prob @ x)

fig, ax = plt.subplots(
    figsize=(6.2, 3.6),
    layout="constrained")
ax.plot(t, x_bar)
ax.axhline(0, color="gray", linewidth=0.7)
ax.set(xlabel="Time",
       ylabel="Mean position",
       xlim=(0, t[-1]))
plt.show()

Find the first broad minimum. Can you distinguish the transfer between wells from the faster fluctuations?

## Check what the mean is telling us

In [ ]:
time_view = 0.0
n = np.argmin(np.abs(t - time_view))

fig, ax = plt.subplots(
    figsize=(6.2, 3.6),
    layout="constrained")
ax.plot(x, prob[n])
ax.set(xlabel="x",
       ylabel="Probability density",
       xlim=(-4, 4),
       ylim=(0, 1.1 * prob.max()),
       title=f"t = {t[n]:.3f}")
plt.show()

Set `time_view` near that minimum. Is most of the probability in the left well, or is it split between the wells?

## How long did the first trip take?

Give the time and mean position at the first major minimum. Use the density at that time to explain whether most of the probability has reached the left well.

For your assigned $\alpha$, check that the run is long enough to see the transfer and return.

Assignment 2 is due Monday, November 2.

Run cells in order. Shift+Enter selects the discussion question on the same slide, leaving the result visible. Click the code cell to edit and run it again; Ctrl+Enter runs without changing the selected cell. After changing `ALPHA`, rerun from the setup onward.